In [ ]:
!pip install -U transformers trl peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.0
    Uninstalling peft-0.21.0:
      Successfully uninstalled peft-0.21.0


In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,Trainer,TrainingArguments
from peft import LoraConfig,get_peft_model,TaskType
from datasets import load_dataset

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
import zipfile
import os
zip_path = "/content/tinyllama-instruction.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"

In [ ]:
!pip install -U torchao
instruction_model = AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [ ]:
inputs = tokenizer(prompt,return_tensors="pt")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens = 180,
    temperature = 0.5,
    do_sample = True,
    top_p = 0.95
)


[transformers] Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:2944: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


In [ ]:
tokenizer.decode(outputs[0],skip_special_tokens=True)

'Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.\nDrug discovery is a process that involves the discovery of new drugs, which are used to treat diseases. Drug discovery is a complex process that involves the collection of data, the analysis of the data, and the synthesis of hypotheses. Drug discovery is a process that involves the synthesis of hypotheses, the analysis of the data, and the collection of data. Drug discovery is a process that involves the synthesis of hypotheses, the analysis of the data, and the collection of data. Drug discovery is a process that involves the synthesis of hypotheses, the analysis of the data, and the collection of data. Drug discovery is a process that involves the synthesis of hypotheses, the analysis of the data, and the collection of data. Drug discovery is a process that involves the synthesis of hypotheses, the analysis of the data'

In [ ]:
!pip install -U trl

In [ ]:
from trl import DPOTrainer,DPOConfig
import torch

In [ ]:
lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = TaskType.CAUSAL_LM,
    target_modules = ["q_proj","k_proj"]
)

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.4 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    use_8bit = True,
)

In [ ]:

instruction_checkpoint = "/content/checkpoint-3"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype = torch.float16,
    device_map = "auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model,instruction_checkpoint)

In [ ]:
model = model.merge_and_unload()

In [ ]:
peft_model_lora = get_peft_model(model,lora_config)

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=8,
    beta=0.1,
    report_to=[],

    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)

In [ ]:
dataset = load_dataset("csv",data_files="/content/pharma_preference_data_200.csv")["train"]

In [ ]:
peft_model_lora.delete_adapter("ref")

In [ ]:
peft_model_lora.peft_config.keys()

dict_keys(['default'])

In [ ]:
trainer = DPOTrainer(
    model=peft_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

Dropping fully truncated examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
10,0.692427
20,0.687276
30,0.679009
40,0.665838
50,0.656694
60,0.639733
70,0.625673
80,0.613959
90,0.605071
100,0.582524


TrainOutput(global_step=200, training_loss=0.5892040061950684, metrics={'train_runtime': 1158.4178, 'train_samples_per_second': 1.381, 'train_steps_per_second': 0.173, 'total_flos': 1500075980292096.0, 'train_loss': 0.5892040061950684, 'epoch': 8.0})

In [ ]:
peft_model_lora.eval()
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."


inputs = {k: v.to("cuda") for k, v in tokenizer(prompt, return_tensors="pt").items()}

with torch.no_grad():
  outputs = peft_model_lora.generate(
      **inputs,
      max_new_tokens = 1024,
      temperature = 0.7,
      do_sample = True,
      top_p = 0.95,
      repetition_penalty = 1.2,
      no_repeat_ngram_size = 3
  )
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
This is an interesting question to discuss as it brings up a very important topic, which has been on the minds of many scientists for quite some time now; that being AI's role in finding new drugs. In today's world, there are countless numbers of scientific theories floating around with no concrete evidence backing them up. There are people who have tried their best, but none have really been able to find out whether or not those theories are right. With this in mind, we can look at what artificial intelligence (AI) could do for the field of chemistry. First off, AI does more than just search databases looking for relevant information about chemical structures. It also helps us understand molecules using different methods such as simulations, pattern recognition and machine learning. This means that AI is able to analyze complex data sets and provide valuable i